# Data preprocessing 

This file presents the preprocessing steps performed from the original raw data available online to the final clean_dataset.csv file used in ActiveTigger. 

## Aggregating raw files into a single CSV

The dataset was downloaded from the Data Assemblée Nationale website, at the URL: https://data.assemblee-nationale.fr/travaux-parlementaires/amendements/tous-les-amendements.      

It came under a zip archive containing folders of XML files, each representing an amendment. 
Using Claude, two Python scripts were generated to (1) unzip the archive (extraire_xml.py) and (2) aggregate the XML files into a single CSV file (extraire_amendements.py).   

A manual check against external sources and the content of the files was made to ensure every piece of information had been correctly transcribed into the CSV file.

## Cleaning CSV file

In [ ]:
import polars as pl

dataset = pl.read_csv("data/amendements.csv")
print(f"There are {dataset.shape[0]} rows and {dataset.shape[1]} columns in the original dataset.")

There are 123224 rows and 25 columns in the original dataset.


### Cleaning variables (columns)

* The text: Based on pre-existing knowledge of the data, we know that sourcing systematically happens in the summary of an amendment, not in the amendment itself. Therefore the text column is *expose_sommaire*.  
* The identifier: The *uid* column contains a unique identifier for each amendment. 
* No context variables.    
* No existing labels.  
All other columns are dropped. 

In [5]:
dataset = dataset.drop(["fichier", "erreur", "legislature", "numero", 
         "texte_legislatif_ref", "auteur_type", "auteur_acteur_ref", 
         "auteur_groupe_politique_ref", "nb_cosignataires", "libelle_signataires", 
         "division_titre", "division_designation", "article_additionnel", "date_depot", 
         "date_publication", "soumis_article_40", "etat_code", "etat_libelle", "sort", 
         "date_sort", "code_loi", "division_code_loi", "dispositif"])

print(f"There are now {dataset.shape[1]} columns: {dataset.columns}.")

There are now 2 columns: ['uid', 'expose_sommaire'].


### Cleaning amendments (rows)

#### Removing duplicates

In [6]:
dataset = dataset.unique(subset="expose_sommaire")
print(f"There are now {dataset.shape[0]} rows.")

There are now 66053 rows.


#### Removing rows with more than 1024 tokens
We use the French-language tokenizer used by the model we intend to choose "almanach/camembertav2-base".

In [7]:
#import HuggingFace's tokenizer and make it run locally
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"  

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("almanach/camembertav2-base")

#deactivate the warning that appears when text is more than 512 tokens long
tokenizer.model_max_length = int(1e9)  

In [8]:
#tokenize data into new column
dataset = dataset.with_columns(pl.col("expose_sommaire")
       .map_elements(lambda x: tokenizer.tokenize(x), return_dtype=pl.List(pl.String))
       .alias("tokenized")
       )

#count number of tokens for each row in "tokenized" in a new column  
dataset = dataset.with_columns(pl.col("tokenized")
       .map_elements(lambda x: len(x))
       .alias("number_tokens")
       )

In [ ]:
#count rows with more than 1024 tokens 
print(f"There are {(dataset.filter(pl.col("number_tokens") > 1024)).shape[0]}"
      " data points with more than 1024 tokens.")

#filter dataset and keep only the rows with 1024 tokens or less
dataset = dataset.filter(pl.col("number_tokens") <= 1024)
print(f"Dataset now has {dataset.shape[0]} rows.")

There are 664 data points with more than 1024 tokens.
Dataset now has 65388 rows.


## Exporting clean CSV

In the end, we have a clean dataset with 62,769 unique amendments of less than 1024 tokens, and only two variables: the text (expose_sommaire) and a unique identifier (uid). 

In [31]:
dataset = dataset.drop(["tokenized", "number_tokens"])
dataset.write_csv("clean_dataset.csv")